# Inference i coalescent modeller

Formålet med denne notebook er at estimere parametre i en coalescent model ud fra observerede data.

Jeg fokuserer på:

1. Method of Moments (MoM)
2. Simulation af data
3. Sammenligning mellem teoretiske og empiriske moments

Dette danner grundlag for senere:

1. inference på rigtige genetiske data
2. mere komplekse modeller (IM, ghost populations)

In [ ]:
from phasic import (
    Graph, with_ipv, GaussPrior, MoMResult, ProbMatchResult,
    Adam, ExpStepSize, clear_caches,
    StateIndexer, Property,
) # ALWAYS import phasic first to set jax backend correctly
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt
import seaborn as sns
%config InlineBackend.figure_format = 'svg'
np.random.seed(42)
try:
    from vscodenb import set_vscode_theme
    set_vscode_theme()
except ImportError:
    pass
sns.set_palette('tab10')

## Method of Moments

Jeg ønsker at estimere parameteren $\theta$ (coalescent rate).

Ide:
Match model moments med data moments:
$$
\hat{\theta}_{\mathrm{MoM}} = \arg\min_{\theta > 0} \left\| m(\theta) - \hat{m} \right\|^2
$$

hvor:

* $m(\theta)$: teoretiske moments fra modellen
* $\hat{m}$: empiriske moments fra data

Fordele:

* hurtigt
* stabilt
* giver gode initial guesses til Bayesian inference

Ulemper:

* ikke altid efficient
* afhænger af hvilke moments man bruger

## Definer model 

Samme model som i notebook 01_basic_coalescent, men nu bare parameteriseret

In [ ]:
nr_samples = 4

@with_ipv([nr_samples] + [0]*(nr_samples-1))
def coalescent(state, theta=1.0):
    transitions = []
    
    for i in range(state.size):
        for j in range(i, state.size):
            same = int(i == j)
            
            if same and state[i] < 2:
                continue
            if not same and (state[i] < 1 or state[j] < 1):
                continue
            
            new = state.copy()
            new[i] -= 1
            new[j] -= 1
            new[i+j+1] += 1
            
            rate = state[i]*(state[j]-same)/(1+same) * theta
            
            transitions.append([new, [rate]])
    
    return transitions


# Graf visualisering 
graph = Graph(coalescent)
graph.plot()

## Simuler data

In [ ]:
true_theta = [7]

graph.update_weights(true_theta)

observed_data = graph.sample(1000)

print("Sample mean:", np.mean(observed_data))

Jeg simulerer data med en kendt parameter $\theta = 7$

Dette gør det muligt at teste hvor godt vores inference metode virker.

## Method of Moments estimation

In [ ]:
mom = graph.method_of_moments(observed_data)

print("True theta:", true_theta)
print("Estimated theta:", mom.theta)
print("Std error:", mom.std)
print("Converged:", mom.success)
print("Residual:", mom.residual)

### Sammenligning af moments 

In [ ]:
print(f"Sample moments: {mom.sample_moments}")
print(f"Model moments:  {mom.model_moments}")

Hvis modellen passer godt:
→ sample moments ≈ model moments

### Visualisering af fit 

In [ ]:
plt.hist(observed_data, bins=50, density=True, alpha=0.5, label="Data")

t = np.linspace(0, 2, 200)
pdf = graph.pdf(t)

plt.plot(t, pdf, label="Model")
plt.legend()
plt.title("Model vs Data")
plt.show()

In [ ]:
prior = mom.prior[0]

prior.plot()

MoM bruges til at konstruere informerede priors:

→ bedre konvergens i Bayesian inference

## SVGD inference

In [ ]:
svgd = graph.svgd(
    observed_data,
    prior=mom.prior,
    learning_rate=ExpStepSize(first_step=0.001, last_step=0.0001, tau=20.0),
)

svgd.summary()
svgd.plot_convergence()

## Konklusion

* Method of Moments giver gode estimater hurtigt
* Estimaterne ligger tæt på den sande værdi
* Metoden er robust og egnet til initialisering af mere avanceret inference
* Dette er fundamentet for senere analyser på rigtige data